# Proces telegraficzny

dr Tomasz Wasak, prof. UMK  
Uniwersytet Mikołaja Kopernika w Toruniu  
Wydział Fizyki, Astronomii i Informatyki Stosowanej  
ul. Grudziądzka 5  
87-100 Toruń  
email: twasak@umk.pl

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

# from numba import jit

# Losowanie przez odrzucanie 

In [ ]:
# @jit(nopython=True)
def draw_p(p, result=None):
    """Zwraca 0 z prawdopodobieństwem p oraz 1 z prawdopodobieństwem 1-p."""

    if result is None:
        result = [0, 1]
    r = np.random.rand()
    if r <= p:
        return result[0]
    elif r >p:
        return result[1]

    # return np.random.choice([0, 1], p=[p, 1 - p])

In [ ]:
Nm = 10000
p = 0.3
draws = [draw_p(p) for x in range(0, Nm)]

In [ ]:
# Plot histogram
plt.hist(draws, bins=4, edgecolor='black', density=False)

# Add labels and title
plt.xlabel('Wartość')
plt.ylabel('Liczebność')
plt.title('Histogram 0 i 1')

# Show the plot
plt.show()

In [ ]:
# Jakie jest prawdopodobieństwo zliczeń 0 i 1?

q_est = np.sum(draws)/Nm
p_est = 1 - q_est
print(f"Prawdopodobieństwo p wystpienia 0 (z histogramu) = {p_est:0.3f}")
print(f"Prawdopodobieństwo p wystpienia 0 (ścisłe)       = {p}")

# Losowanie trajektorii

In [ ]:
# Definiujemy funkcję generującą trajektorię w procesie telegraficznym

def generate_trajectory(x0, n_steps, p):
    trajectory = np.zeros(n_steps)
    trajectory[0] = x0
    
    for i in range(1, n_steps):
        trajectory[i] = trajectory[i-1]*draw_p(p, result=[1,-1])  # draw the sign (+/-1) with prob. p

    return trajectory

Definiujemy parametry wchodzące do problemu.

In [ ]:
dt = 0.1                                # krok czasowy
gamma = 1.0                             # stała zaniku gamma
p = 1/2 * (1 + np.exp(-2*gamma*dt))     # prawdopodobieństwo pozostania w stanie
nsteps = 1000                           # liczba kroków czasowych

# generujemy oś czasu: 
ts = dt*np.arange(0, nsteps)

print(f"Prawdopodobieństwo pozostania w stanie p = {p:0.3f}")
print(f"Prawdopodobieństwo przeskokou 1-p        = {1-p:0.3f}")

In [ ]:
traj = generate_trajectory(1, nsteps, p)
#traj

In [ ]:
plt.plot(ts, traj, ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"Stan, $X(t)$")
plt.title("Jedna trajektoria")
#plt.xlim(0,10)
plt.show()

# Generujemy trajektorie - wiele trajektorii

Generujemy wiele trajektorii, potem będziemy badać ich statystykę.

In [ ]:
%%time

ntrajs = 1000
trajs = [generate_trajectory(draw_p(0.5, [1,-1]), nsteps, p) for _ in range(ntrajs)]  # start with 1/2 random +/-1
trajs = np.array(trajs)
#traj

Jak wygląda jedna trajektoria?

In [ ]:
idx = 13

plt.plot(ts, trajs[idx], ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"Stan, $X(t)$")
#plt.xlim(0,20)
plt.title(f"Jedna trajektoria nr {idx}")
plt.show()

# Uśrednianie 

Policzmy teraz funkcję korelacji $\kappa(t) = \langle x(t)x(0) \rangle$

In [ ]:
def kappa(n0, n1, trajs):
    res = 0
    for tr in trajs:
        res += tr[n0]*tr[n1]

    return res/len(trajs)

In [ ]:
kappa(0,0, trajs)

In [ ]:
ts[10], kappa(0,10, trajs)

In [ ]:
t_idx = 100

print(f"Sprawdźmy funkcję korelacji dla t = {dt*t_idx}")
print(f"t = {ts[t_idx]}, \nkappa = {kappa(0,t_idx, trajs)}")

In [ ]:
kappas = [ kappa(0, idxt, trajs) for idxt, t in enumerate(ts) ]

In [ ]:
plt.plot(ts, kappas, ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"$\kappa(t)$")
plt.title(r"Funkcja korelacji $\kappa(t)$")
#plt.xlim(0,10)
plt.show()

# Porównanie analitycznych obliczeń z numerycznymi

In [ ]:
def kappa_theo(t, gamma):
    return np.exp(- 2*gamma*t)

In [ ]:
kappas_theo = kappa_theo(ts, gamma)

In [ ]:
plt.plot(ts, kappas, ".-")
plt.plot(ts, kappas_theo, ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"$\kappa(t)$")
plt.title(f"Funkcja korelacji")
plt.xlim(0,14)
plt.show()

# Jaka jest średnia $\langle X(t)\rangle$ oraz $\langle X^2(t)\rangle$?

In [ ]:
def x_average(t_idx, trajs):
    res = 0

    for tr in trajs:
        res += tr[t_idx]

    return res/len(trajs)


def x2_average(t_idx, trajs):
    res = 0

    for tr in trajs:
        res += tr[t_idx]**2

    return res/len(trajs)

In [ ]:
xs_average = [x_average(tidx, trajs) for tidx, t in enumerate(ts)]
x2s_average = [x2_average(tidx, trajs) for tidx, t in enumerate(ts)]

In [ ]:
plt.plot(ts, xs_average, ".-")
plt.plot(ts, x2s_average, ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"$\langle X(t) \rangle $")
plt.title(f"Funkcja korelacji")
#plt.xlim(0,14)
plt.ylim(-1.3,1.3)
plt.show()

# Jaka jest średnia $\langle X(t)\rangle$ jeśli w $t=t_0$ zmierzyliśmy $+1$?

In [ ]:
# szukamy takich trajektorii dla których w czasie t_idx trajektoria ma X(t) = +1
def find_subensemble(trajs, t_idx, result_condition):
    ensemble_ids = []

    for i,tr in enumerate(trajs):
        if tr[t_idx] == result_condition:
            ensemble_ids.append(i)
    return ensemble_ids

In [ ]:
# Najpierw wybieramy te trajektorie, które w t0 zmierzyły +1

tidx_subensemble = 50  # indeks t0 
subensemble = find_subensemble(trajs, tidx_subensemble, +1)

In [ ]:
random_trajectory_nr = 15

val = trajs[subensemble[random_trajectory_nr]][tidx_subensemble]
print(f"Czas:       t_0 = {dt*tidx_subensemble}")
print(f"Wartość: X(t_0) = {val}")

In [ ]:
# nowa zmienna opisująca trajektorie tylko z podzbioru ("pod-zespołu" = sub-ensemble)
trajs_subensemble = [trajs[idx] for idx in subensemble]

In [ ]:
# tworzymy średnią X(t) wśród trajektorii, które mają X(t) = +1 dla t = t_0.
xs_average_subensemble = [x_average(tidx, trajs_subensemble) for tidx, t in enumerate(ts)]

In [ ]:
plt.plot(ts, xs_average_subensemble, ".-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"$\langle X(t) \rangle $")
plt.title(f"Zmierzono X(t) = +1 dla t = {dt*tidx_subensemble}")
#plt.xlim(0,14)
plt.ylim(-1.3,1.3)
plt.show()

In [ ]:
# Teoretyczne przewidywanie
def x_average_condition_theory(t, t0, gamma):
    return np.exp(-2 * gamma * np.abs(t-t0))

In [ ]:
plt.plot(ts, xs_average_subensemble, ".-")
plt.plot(ts, x_average_condition_theory(ts, tidx_subensemble*dt, gamma), "-")
plt.xlabel(r"Czas, $t$")
plt.ylabel(r"$\langle X(t) \rangle $")
plt.title(f"Zmierzono X(t) = +1 dla t = {dt*tidx_subensemble}")
plt.xlim(0,20)
plt.ylim(-1.3,1.3)
plt.show()

# Jaki jest średni czas przełaczenia z $+1 \to  -1$ lub $-1 \to  +1$?

In [ ]:
# +1 oznacza przeskok stanu
switching = np.abs(np.diff(trajs[0]))/2

In [ ]:
plt.plot(ts, trajs[0])
plt.plot(ts[:-1], switching)
plt.xlim(0,10)
plt.show()

In [ ]:
duration = []
res = 0

# mierzymy odstęp czasu, aż do zmiany, potem zerujemy licznik
for idx, i in enumerate(switching):
    res += 1
    if switching[idx]==1:
        duration.append(res*dt)
        res=0

In [ ]:
duration = np.array(duration)

In [ ]:
# średni czas pomiędzy przełączeniami
np.average(duration)

In [ ]:
# automatyzacja
# wyliczamy czas trwania stanu aż do przeskoku

def switching_times(traj):
    switching = np.abs(np.diff(traj))/2

    duration = []
    res = 0
    
    # mierzymy odstęp czasu, aż do zmiany, potem zerujemy licznik
    for idx, i in enumerate(switching):
        if switching[idx]==1:
            duration.append(res*dt)
            res=0
            continue

        res += 1

    return np.array(duration)

In [ ]:
np.average(switching_times(trajs[0]))  # średnia po jednej trajektorii

In [ ]:
# czasy przełaczani dla kolejnych trajektorii
switching_times_tab = [switching_times(tr) for tr in trajs]

In [ ]:
# usrednianie po trajektoriach - n-te przełaczenie
def nth_switching_time(n, switching_times_tab):
    res = 0
    for idx, st in enumerate(switching_times_tab):
        res += st[n]

    return res / len(switching_times_tab)

In [ ]:
# 0-th switching
nth_switching_time(0, switching_times_tab)

In [ ]:
# 1-st
nth_switching_time(1, switching_times_tab)

In [ ]:
# 2-nd
nth_switching_time(2, switching_times_tab)

In [ ]:
# 3-rd
nth_switching_time(3, switching_times_tab)

In [ ]:
# 4-rd
nth_switching_time(4, switching_times_tab)